In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [ ]:
def qrng(n):
    qc = QuantumCircuit(n, n)
    for i in range(n):
        qc.h(i)
    qc.measure(range(n), range(n))
    backend = BasicSimulator()
    job = backend.run(transpile(qc, backend), shots=1)
    counts = job.result().get_counts()
    bits = list(counts.keys())[0][::-1]
    return [int(b) for b in bits]

N = 100

print("testing qrng:", qrng(8))

In [ ]:
alice_bits = qrng(N)
alice_bases = qrng(N)

print("Alice's bits: ", alice_bits)
print("Alice's bases:", alice_bases)

In [ ]:
eve_bases = qrng(N)

qc_eve = QuantumCircuit(N, N)

for i in range(N):
    if alice_bits[i] == 1:
        qc_eve.x(i)
    if alice_bases[i] == 1:
        qc_eve.h(i)

for i in range(N):
    if eve_bases[i] == 1:
        qc_eve.h(i)

qc_eve.measure(range(N), range(N))

backend = BasicSimulator()
job_eve = backend.run(transpile(qc_eve, backend), shots=1)
eve_bits = list(job_eve.result().get_counts().keys())[0][::-1]
eve_bits = [int(b) for b in eve_bits]

print("Eve's bases:", eve_bases)
print("Eve's bits: ", eve_bits)

In [ ]:
bob_bases = qrng(N)

qc_bob = QuantumCircuit(N, N)

for i in range(N):
    if eve_bits[i] == 1:
        qc_bob.x(i)
    if eve_bases[i] == 1:
        qc_bob.h(i)

for i in range(N):
    if bob_bases[i] == 1:
        qc_bob.h(i)

qc_bob.measure(range(N), range(N))

job_bob = backend.run(transpile(qc_bob, backend), shots=1)
bob_bits = list(job_bob.result().get_counts().keys())[0][::-1]
bob_bits = [int(b) for b in bob_bits]

print("Bob's bases:", bob_bases)
print("Bob's bits: ", bob_bits)

In [ ]:
alice_key = []
bob_key = []

for i in range(N):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_bits[i])

print("Alice's key:", alice_key)
print("Bob's key:  ", bob_key)
print("Key length:", len(alice_key), "out of", N, "qubits")

In [ ]:
half = len(alice_key) // 2
check_alice = alice_key[:half]
check_bob = bob_key[:half]

errors = 0
for i in range(half):
    if check_alice[i] != check_bob[i]:
        errors += 1

error_rate = errors / half
threshold = 0.11

print("Errors:", errors, "out of", half, "bits checked")
print("Error rate:", round(error_rate * 100, 1), "%")
print()

if error_rate > threshold:
    print("Attack detected! Error rate is above", threshold * 100, "% threshold")
else:
    print("No attack detected")